# Neuron Flow 0.4.0.dev0 — หลายสกุลเงิน

รันใน runtime ใหม่: ติดตั้ง ZIP → อัปโหลด CSV → บันทึกรหัส batch ก่อนทำงาน → ดูรายงาน → ทำต่อ → ตรวจทั้งชุด → ดาวน์โหลด backup
รองรับ 46 สกุลเงิน ใช้จำนวนตำแหน่งทศนิยมตามสกุล และแยกยอดรวมรายสกุลโดยไม่แปลงอัตราแลกเปลี่ยน
ผล Colab จริงยังเป็น not_run จนมีหลักฐานครบ การตรวจในเครื่องและการเผยแพร่ release ไม่ใช้แทนผล Colab

เปิดงาน CSV 0.2.0/0.2.1 ต่อด้วย 0.4.0.dev0 ได้เมื่อ schema/contract/Agent เข้ากันได้ โดยไม่เปลี่ยนประวัติและรายงานเดิม
หากหยุดก่อนเห็นผล รหัส batch ถูกบันทึกแล้ว หรือค้นจาก workspace ด้วย batch-list ได้
ชุดอ้างอิงเดิม 100 เอกสารยังตรวจแบบ v1 (80 VALID / 20 INVALID); การรองรับสกุลเงินใหม่ตรวจแยก และจำนวน tests อ่านจากรายงานของรอบนี้

สำหรับตรวจรับจริงให้ทำตาม docs/COLAB_ACCEPTANCE_TH.md และใช้ docs/colab_acceptance_TEMPLATE.json เก็บหลักฐาน
ต้องยืนยันว่าได้รับไฟล์ดาวน์โหลดจริงและคืน backup ใน runtime ใหม่ได้ จึงจะบันทึกว่าตรวจ Colab ครบ


## 1. อัปโหลด ZIP รุ่น 0.4.0.dev0 และตรวจ SHA-256

เลือก **neuron_flow_v0.4.0.dev0.zip เพียงไฟล์เดียว** ระบบตรวจเส้นทาง ZIP, ขนาด และ SHA-256
ทุกไฟล์ก่อนเรียกโค้ดโปรเจกต์ ใช้ Python standard library; ไม่ต้องติดตั้ง dependency เพิ่ม
รันเซลล์นี้ครั้งเดียวต่อ runtime เพื่อรักษาไฟล์และผลตรวจเดิม


In [ ]:
from google.colab import files
from pathlib import Path, PurePosixPath
from datetime import datetime, timezone
from contextlib import closing
import ast
import csv
import hashlib
import io
import json
import os
import platform
import shutil
import sqlite3
import stat
import subprocess
import sys
import tempfile
import zipfile

if sys.version_info < (3, 10):
    raise RuntimeError('ต้องใช้ Python 3.10 ขึ้นไป')

PROJECT = Path('/content/neuron_flow_project/neuron_flow')
WORKSPACE = Path('/content/neuron_flow_workspace')
SESSION_PATH = Path('/content/neuron_flow_session.json')
UPLOAD_DIRECTORY = Path('/content/neuron_flow_uploads')
OBSERVATIONS = {'runtime': 'Google Colab', 'python': sys.version, 'platform': platform.platform(),
                'started_at_utc': datetime.now(timezone.utc).isoformat(),
                'download_received_confirmed': False,
                'runtime_marker_names': [name for name in ('COLAB_RELEASE_TAG', 'COLAB_BACKEND_VERSION', 'COLAB_JUPYTER_TRANSPORT') if name in os.environ]}

def safe_unpack(data, destination, allowed_roots):
    if len(data) > 64 * 1024 * 1024:
        raise ValueError('ZIP ต้องไม่เกิน 64 MiB')
    destination = Path(destination)
    destination.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(io.BytesIO(data)) as archive:
        entries = archive.infolist()
        if len(entries) > 5000 or sum(item.file_size for item in entries) > 256 * 1024 * 1024:
            raise ValueError('ZIP มีจำนวนไฟล์หรือขนาดหลังแตกเกินขอบเขต')
        seen = set()
        for item in entries:
            relative = PurePosixPath(item.filename)
            mode = (item.external_attr >> 16) & 0xffff
            if (not relative.parts or relative.is_absolute() or '..' in relative.parts
                    or '\\' in item.filename or any(':' in part for part in relative.parts)
                    or relative.parts[0] not in allowed_roots
                    or stat.S_IFMT(mode) not in (0, stat.S_IFREG, stat.S_IFDIR)
                    or item.flag_bits & 1 or item.file_size > 64 * 1024 * 1024):
                raise ValueError('พบเส้นทางหรือชนิดไฟล์ที่ไม่รองรับใน ZIP')
            target = (destination / item.filename).resolve()
            if not target.is_relative_to(destination.resolve()) or target in seen:
                raise ValueError('ZIP มีเส้นทางซ้ำหรืออยู่นอกโฟลเดอร์ปลายทาง')
            seen.add(target)
        archive.extractall(destination)

def verify_sums(root, manifest_name='SHA256SUMS.txt'):
    root = Path(root)
    checked = set()
    for line in (root / manifest_name).read_text(encoding='utf-8').splitlines():
        digest, separator, name = line.partition('  ')
        relative = PurePosixPath(name)
        if (not separator or len(digest) != 64 or any(c not in '0123456789abcdef' for c in digest)
                or not relative.parts or relative.is_absolute() or '..' in relative.parts
                or '\\' in name or name == manifest_name or name in checked):
            raise ValueError('รูปแบบ SHA-256 manifest ไม่ถูกต้อง')
        path = root / name
        if (not path.resolve().is_relative_to(root.resolve()) or not path.is_file()
                or path.is_symlink() or hashlib.sha256(path.read_bytes()).hexdigest() != digest):
            raise ValueError('SHA-256 ไม่ตรง: ' + name)
        checked.add(name)
    actual = {p.relative_to(root).as_posix() for p in root.rglob('*')
              if p.is_file() and p.relative_to(root).as_posix() != manifest_name}
    if not checked or checked != actual:
        raise ValueError('Manifest ไม่ครอบคลุมไฟล์ทั้งหมด')
    return len(checked)

if PROJECT.exists():
    raise RuntimeError('มีโปรเจกต์ใน runtime นี้แล้ว ใช้เซลล์ถัดไป หรือเริ่ม runtime ใหม่ก่อนติดตั้ง ZIP')
uploaded = files.upload()
if len(uploaded) != 1 or not next(iter(uploaded)).lower().endswith('.zip'):
    raise ValueError('เลือก ZIP ของโปรเจกต์เพียงไฟล์เดียว')
safe_unpack(next(iter(uploaded.values())), PROJECT.parent, {'neuron_flow'})
verified_files = verify_sums(PROJECT)
required = ['neuron_flow/__main__.py', 'neuron_flow/batch.py', 'neuron_flow/reporting.py', 'neuron_flow/currencies.py',
            'tools/verify_v020.py', 'acceptance/fixture_lock.json']
if not all((PROJECT / name).is_file() for name in required):
    raise ValueError('ชุดโปรเจกต์ไม่ครบ')
version = subprocess.check_output([sys.executable, '-c',
    'from neuron_flow import __version__; print(__version__)'], cwd=PROJECT, text=True).strip()
if version != '0.4.0.dev0':
    raise ValueError('ZIP ต้องเป็นรุ่น 0.4.0.dev0')
LEGACY_020_MANIFEST_SHA = '0af9c310e75717386bcf2cd4cc00a5eeb92a8c5de5ee819bd030681d2ce2f41a'
PACKAGE_MANIFEST_SHA = hashlib.sha256((PROJECT / 'SHA256SUMS.txt').read_bytes()).hexdigest()
OBSERVATIONS.update(project_version=version, project_files_verified=verified_files,
                    package_manifest_sha256=PACKAGE_MANIFEST_SHA)

def run_cli(*arguments):
    completed = subprocess.run([sys.executable, '-m', 'neuron_flow', *map(str, arguments)],
                               cwd=PROJECT, capture_output=True, text=True, timeout=300)
    if completed.stderr:
        print(completed.stderr)
    if completed.returncode not in (0, 1):
        print(completed.stdout)
        completed.check_returncode()
    result = json.loads(completed.stdout)
    print('exit code:', completed.returncode, 'สถานะ:', result.get('batch_status', result.get('status')))
    if 'batches' in result:
        print('batch ที่พบ:', [(item['batch_id'], item['status']) for item in result['batches']])
    else:
        print('จำนวนเอกสาร:', result.get('counts', {}), 'สกุลเงิน:', result.get('currency'))
        print('ยอดที่รับรอง:', result.get('certified_totals', result.get('certified_total')))
    if completed.returncode == 1:
        print('พบ ERROR หรือส่งมอบยังไม่สำเร็จ โปรดตรวจผลรายเอกสารและ receipt')
    return result

print('พร้อมใช้:', PROJECT, '| Python', platform.python_version(), '| ตรวจ SHA-256 แล้ว', verified_files, 'ไฟล์')


## 2. อัปโหลด CSV และประมวลผล

เลือก CSV **หนึ่งไฟล์** UTF-8 หรือ UTF-8 BOM ไม่เกิน 16 MiB ใช้ชื่อช่อง
`document_id,item_id,description,quantity,unit_price,declared_total,currency`
หากต้องการตัวอย่าง ไฟล์ `examples/purchases.csv`, `examples/purchases_invalid.csv` และ `examples/purchases_international.csv` อยู่ใน ZIP
แต่ละเอกสารต้องใช้สกุลเดียวกัน ดูรหัสทั้ง 46 สกุลและกติกาทศนิยมใน docs/CURRENCIES_TH.md
รหัส batch สร้างจากเนื้อหา/รุ่นโดยอัตโนมัติ ข้อมูลผิดจะเป็น INVALID แยกใบ ส่วน ERROR หมายถึงระบบยืนยันผลไม่ได้
ยอดที่รับรองรวมเฉพาะ VALID แยกตามสกุล ไม่มีการรวมเงินต่างสกุลเป็นยอดเดียว; แถวไม่มีรหัสเก็บใน ungrouped_rows

เมื่อคืน backup ให้ข้ามเซลล์นี้และไปข้อ 4


In [ ]:
uploaded_csv = files.upload()
if len(uploaded_csv) != 1 or not next(iter(uploaded_csv)).lower().endswith('.csv'):
    raise ValueError('เลือก CSV เพียงไฟล์เดียว')
csv_bytes = next(iter(uploaded_csv.values()))
if len(csv_bytes) > 16 * 1024 * 1024:
    raise ValueError('CSV ต้องไม่เกิน 16 MiB')
upload_directory = UPLOAD_DIRECTORY
upload_directory.mkdir(exist_ok=True)
source_path = upload_directory / 'input.csv'
source_path.write_bytes(csv_bytes)
SUBMISSION = run_cli('csv', source_path, '--submit-only', '--workspace', WORKSPACE)
BATCH_ID = SUBMISSION['batch_id']
SESSION_PATH.write_text(json.dumps({'batch_id': BATCH_ID, 'workspace': str(WORKSPACE),
    'project_version': version, 'package_manifest_sha256': PACKAGE_MANIFEST_SHA},
    ensure_ascii=False, indent=2), encoding='utf-8')
OBSERVATIONS.update(csv_uploaded=True, csv_sha256=hashlib.sha256(csv_bytes).hexdigest(),
                    batch_id=BATCH_ID, batch_persisted_before_execution=True)
print('บันทึกรหัส batch ก่อนประมวลผลแล้ว:', BATCH_ID)
RESULT = run_cli('batch-resume', BATCH_ID, '--workspace', WORKSPACE)
OBSERVATIONS.update(cli_counts=RESULT.get('counts'), cli_executed=True)
print('แถวที่จัดกลุ่มไม่ได้:', RESULT.get('ungrouped_rows'))
print(json.dumps(RESULT.get('receipt', {}), ensure_ascii=False, indent=2))


## 3. ดูตัวอย่างรายงาน

ผลรายใบมี VALID / INVALID / ERROR และ engine_status แยกกัน การส่งรายงานสำเร็จยังต้องแสดง ERROR ทุกใบ
source_rows นับบรรทัดจริงเริ่มต้น record; ข้อความ CSV หลายบรรทัดอาจทำให้เลขแถวไม่เรียงติดกัน


In [ ]:
session = json.loads(SESSION_PATH.read_text(encoding='utf-8'))
BATCH_ID = session['batch_id']
STATUS = run_cli('batch-status', BATCH_ID, '--workspace', WORKSPACE)
artifact_info = (STATUS.get('receipt') or {}).get('artifacts', {})
if not artifact_info:
    raise RuntimeError('ยังไม่มีรายงานครบชุด ให้ตรวจสถานะและใช้เซลล์ทำต่อ')
summary_path = Path(artifact_info['summary.csv']['path'])
with summary_path.open(encoding='utf-8', newline='') as handle:
    preview = list(csv.DictReader(handle))[:10]
print(json.dumps(preview, ensure_ascii=False, indent=2))
details = json.loads(Path(artifact_info['details.json']['path']).read_bytes())
print('ข้อมูลระดับชุด:', {key: details.get(key) for key in
    ('counts', 'currency', 'certified_total', 'certified_totals', 'source_sha256')})
print('ungrouped_rows:', len(details['ungrouped_rows']))
OBSERVATIONS['report_previewed'] = True


## 4. ทำต่อโดยไม่อัปโหลดต้นทางซ้ำ

ใช้ workspace เดิม ระบบเก็บ CSV ใน SQLite แล้ว ใช้ผลสำเร็จเดิมและไม่สร้างรายงานเดิมซ้ำ
ค่าเริ่มต้นลองรวมไม่เกิน 2 ครั้ง; ใบที่หมดงบยังเป็น ERROR งานสำเร็จไม่ถูกลองใหม่
หากคืน backup ใน runtime ใหม่แล้ว รันเซลล์นี้ได้ทันที


In [ ]:
if not SESSION_PATH.is_file():
    found = run_cli('batch-list', '--workspace', WORKSPACE)['batches']
    if not found:
        raise RuntimeError('ไม่พบ batch ใน workspace นี้')
    identifiers = [item['batch_id'] for item in found]
    chosen = identifiers[0] if len(identifiers) == 1 else input('กรอกรหัส batch ที่ต้องการทำต่อ: ').strip()
    if chosen not in identifiers:
        raise ValueError('รหัส batch ไม่อยู่ในรายการ')
    SESSION_PATH.write_text(json.dumps({'batch_id': chosen, 'workspace': str(WORKSPACE),
        'project_version': version, 'package_manifest_sha256': PACKAGE_MANIFEST_SHA},
        ensure_ascii=False, indent=2), encoding='utf-8')
    OBSERVATIONS['batch_discovery_used'] = True
session = json.loads(SESSION_PATH.read_text(encoding='utf-8'))
if session['workspace'] != str(WORKSPACE) or not (WORKSPACE / 'state.sqlite3').is_file():
    raise RuntimeError('ไม่พบ workspace เดิม ให้คืน backup ก่อน')
BATCH_ID = session['batch_id']
RESULT = run_cli('batch-resume', BATCH_ID, '--workspace', WORKSPACE)
OBSERVATIONS.update(resume_executed=True, resume_counts=RESULT.get('counts'), batch_id=BATCH_ID)
print(json.dumps(RESULT.get('receipt', {}), ensure_ascii=False, indent=2))


## 5. ตรวจฐานเดิมและชุดปัจจุบัน พร้อมเก็บหลักฐาน

คำสั่งนี้แยก 38 กรณีฐานเดิมจากชุดใหม่ และรัน 100 เอกสารเทียบคำตอบที่ล็อก
รวมกรณี controller หยุดจริง การเปิด workspace CSV 0.2.0/0.2.1 ด้วย 0.4.0.dev0 ขอบเขตข้อมูล และการค้นหา batch
จำนวนกรณีใหม่อ่านจากผลจริง ไม่กำหนดตามจำนวนของรุ่นก่อน

ผลสำเร็จในเครื่องไม่ทำให้ Colab acceptance สมบูรณ์อัตโนมัติ ต้องเก็บหลักฐานตาม checklist รวมการได้รับไฟล์ดาวน์โหลดและคืน backup จริง
การอนุมัติเผยแพร่รุ่นนี้บันทึกแยกใน RELEASE_APPROVAL.json โดยไม่เปลี่ยนสถานะตรวจ Colab ให้ผ่าน


In [ ]:
checked = subprocess.run([sys.executable, 'tools/verify_current.py'], cwd=PROJECT,
                         capture_output=True, text=True, timeout=300)
print(checked.stdout)
if checked.stderr:
    print(checked.stderr)
checked.check_returncode()
verification = json.loads((PROJECT / 'verification/v0_3_0/test_results.json').read_text(encoding='utf-8'))
print(json.dumps({key: verification[key] for key in
    ('version', 'baseline_38_passed', 'new_test_cases', 'failures', 'errors', 'skipped',
     'local_successful', 'colab', 'colab_acceptance_complete', 'release_ready', 'remaining_acceptance')}, ensure_ascii=False, indent=2))
OBSERVATIONS.update(verifier_executed=True, verifier_local_successful=verification['local_successful'],
                    observed_colab=verification['colab'])


## 6. ดาวน์โหลด workspace และหลักฐาน

ต้องรอให้คำสั่งก่อนหน้าจบก่อนดาวน์โหลด เซลล์สำรอง SQLite ด้วย backup API แล้วรวมรายงาน
verification และ session metadata พร้อม SHA-256 เก็บ ZIP นี้ไว้ เพราะ Colab ลบไฟล์เมื่อ session สิ้นสุด
ตรวจว่าเบราว์เซอร์ได้รับไฟล์จริง การเรียก `files.download` อย่างเดียวยังไม่ยืนยันว่าดาวน์โหลดถึงเครื่องแล้ว

การคืน backup ต้องใช้โปรเจกต์ ZIP เดิมและ path เดิม `/content/neuron_flow_workspace`
เพราะ receipt ใน SQLite เก็บ absolute path ไฟล์หลักฐาน session จะไม่อ้างว่าดาวน์โหลดสำเร็จโดยอัตโนมัติ


In [ ]:
if not (WORKSPACE / 'state.sqlite3').is_file() or not SESSION_PATH.is_file():
    raise RuntimeError('ยังไม่มี workspace/session ให้สำรอง')
OBSERVATIONS['backup_prepared_at_utc'] = datetime.now(timezone.utc).isoformat()
observation_path = PROJECT / 'verification/v0_3_0/colab_notebook_session.json'
observation_path.parent.mkdir(parents=True, exist_ok=True)
observation_path.write_text(json.dumps(OBSERVATIONS, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
stage = Path(tempfile.mkdtemp(prefix='neuron_flow_backup_', dir='/content'))
shutil.copytree(WORKSPACE, stage / 'workspace', ignore=shutil.ignore_patterns(
    'state.sqlite3', 'state.sqlite3-wal', 'state.sqlite3-shm', '*.tmp'))
with closing(sqlite3.connect('file:' + str(WORKSPACE / 'state.sqlite3') + '?mode=ro', uri=True)) as original_db:
    with closing(sqlite3.connect(stage / 'workspace/state.sqlite3')) as backup_db:
        original_db.backup(backup_db)
shutil.copytree(PROJECT / 'verification', stage / 'verification')
shutil.copy2(SESSION_PATH, stage / 'session.json')
manifest = []
for path in sorted(stage.rglob('*')):
    if path.is_symlink():
        raise ValueError('พบ symlink ใน workspace ที่จะสำรอง')
    if path.is_file():
        manifest.append(hashlib.sha256(path.read_bytes()).hexdigest() + '  ' + path.relative_to(stage).as_posix())
(stage / 'backup_SHA256SUMS.txt').write_text('\n'.join(manifest) + '\n', encoding='utf-8')
verify_sums(stage, 'backup_SHA256SUMS.txt')
backup_path = Path('/content/neuron_flow_workspace_backup.zip')
with zipfile.ZipFile(backup_path, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(stage.rglob('*')):
        if path.is_file():
            archive.write(path, path.relative_to(stage).as_posix())
print('backup ZIP SHA-256:', hashlib.sha256(backup_path.read_bytes()).hexdigest())
print('สำรองแล้ว:', backup_path, '| bytes:', backup_path.stat().st_size)
files.download(str(backup_path))


## ทางเลือก: คืน backup ใน Colab runtime ใหม่

ติดตั้ง ZIP 0.4.0.dev0 ก่อน แล้วเลือก ZIP backup เพียงไฟล์เดียว จากนั้นไปข้อ 4 เพื่อทำต่อ
รองรับ backup 0.2.0 ที่สร้างจากชุดเผยแพร่เดิมซึ่งล็อก manifest SHA ไว้ หรือ backup ของ ZIP 0.4.0.dev0 ชุดเดียวกัน
ทั้งสองกรณีต้องคืน path เดิม ระบบไม่เขียนทับ workspace ที่มีข้อมูล และตรวจ contract/Agent/identity อีกครั้งก่อนเริ่มงาน

ตัว runtime เปิด workspace CSV 0.2.1 ที่เข้ากันได้ต่อได้ แต่เซลล์คืน backup นี้ไม่ถือว่า hash ของ ZIP 0.4.0.dev0 ยืนยัน backup 0.2.1 ได้
ให้ใช้ Notebook/ZIP 0.2.1 เดิมสำหรับคืน backup รุ่นนั้น การคืน backup ไปคนละโฟลเดอร์ยังไม่รองรับ
งาน invoice JSON ที่ใช้ Agent รุ่น 1 ต้องใช้ runtime เดิมหรือสร้าง job ใหม่ เนื่องจากกติกาคำนวณเปลี่ยนเป็น Agent รุ่น 2


In [ ]:
if WORKSPACE.exists() and any(WORKSPACE.iterdir()):
    raise RuntimeError('workspace มีข้อมูลอยู่แล้ว จึงไม่คืน backup ทับ')
restored_upload = files.upload()
if len(restored_upload) != 1 or not next(iter(restored_upload)).lower().endswith('.zip'):
    raise ValueError('เลือก ZIP backup เพียงไฟล์เดียว')
restore_stage = Path(tempfile.mkdtemp(prefix='neuron_flow_restore_', dir='/content'))
safe_unpack(next(iter(restored_upload.values())), restore_stage,
            {'workspace', 'verification', 'session.json', 'backup_SHA256SUMS.txt'})
verify_sums(restore_stage, 'backup_SHA256SUMS.txt')
restored_session = json.loads((restore_stage / 'session.json').read_text(encoding='utf-8'))
backup_version = restored_session['project_version']
expected_manifest = {'0.2.0': LEGACY_020_MANIFEST_SHA, version: PACKAGE_MANIFEST_SHA}.get(backup_version)
if (restored_session['workspace'] != str(WORKSPACE) or expected_manifest is None
        or restored_session['package_manifest_sha256'] != expected_manifest):
    raise ValueError('backup ต้องใช้ path เดิมและชุด ZIP 0.2.0 ที่รองรับหรือ ZIP รุ่นปัจจุบันชุดเดียวกัน')
if not (restore_stage / 'workspace/state.sqlite3').is_file():
    raise ValueError('backup ไม่มี SQLite workspace')
shutil.copytree(restore_stage / 'workspace', WORKSPACE, dirs_exist_ok=True)
shutil.copytree(restore_stage / 'verification', PROJECT / 'verification', dirs_exist_ok=True)
restored_session.update(restored_from_version=backup_version, project_version=version,
                        package_manifest_sha256=PACKAGE_MANIFEST_SHA)
SESSION_PATH.write_text(json.dumps(restored_session, ensure_ascii=False, indent=2), encoding='utf-8')
BATCH_ID = restored_session['batch_id']
OBSERVATIONS.update(backup_restored=True, batch_id=BATCH_ID)
print('คืน workspace แล้ว:', WORKSPACE, '| batch_id:', BATCH_ID)
print('ไปข้อ 4 เพื่อทำต่อโดยไม่อัปโหลด CSV ซ้ำ')
